In [1]:
import os
from build123d import *
import plotly.graph_objects as go

# -----------------------------------------------------------------------------
# Parameters (from PN_000669_v9_export.json)
# -----------------------------------------------------------------------------
PART_NAME = "PN_000669_v9"

# Dimensions in mm
WIDTH = 15.5
DEPTH = 10.0
THICKNESS = 3.175
CORNER_R = 1.0
HOLE_DIA = 3.2
HOLE_X_OFFSET = 3.8  # d8 parameter

# -----------------------------------------------------------------------------
# Build
# -----------------------------------------------------------------------------
def build_component():
    with BuildPart() as component:
        # Main plate on XZ plane as per Sketch1
        with BuildSketch(Plane.XZ) as sk:
            Rectangle(WIDTH, DEPTH)
            fillet(sk.vertices(), radius=CORNER_R)
            
            # Holes
            with Locations((HOLE_X_OFFSET, 0), (-HOLE_X_OFFSET, 0)):
                Circle(HOLE_DIA / 2, mode=Mode.SUBTRACT)
        
        # Extrude along Y axis (normal to XZ plane)
        extrude(sk.sketch, amount=THICKNESS)
        
    return component.part

# -----------------------------------------------------------------------------
# Analysis & Export
# -----------------------------------------------------------------------------
def analyze_and_export(part):
    desktop = os.path.expanduser("~/Desktop")
    stl_path = os.path.join(desktop, f"{PART_NAME}.stl")
    step_path = os.path.join(desktop, f"{PART_NAME}.step")
    
    export_stl(part, stl_path)
    export_step(part, step_path)
    
    print(f"--- {PART_NAME} Analysis ---")
    print(f"Volume: {part.volume:.3f} mm^3")
    bb = part.bounding_box()
    print(f"Bounding Box: X[{bb.min.X:.2f}, {bb.max.X:.2f}], Y[{bb.min.Y:.2f}, {bb.max.Y:.2f}], Z[{bb.min.Z:.2f}, {bb.max.Z:.2f}]")
    
    # Volumetric difference
    ref_step = f"/Users/softage/Desktop/Thelio/PN_000669 v9.step"
    if os.path.exists(ref_step):
        ref_part = import_step(ref_step)
        diff = abs(part.volume - ref_part.volume)
        print(f"Reference Volume: {ref_part.volume:.3f} mm^3")
        print(f"Volumetric Difference: {diff:.3f} mm^3 ({(diff/ref_part.volume)*100:.2f}%)")
    else:
        print(f"Note: Reference STEP file '{ref_step}' not found.")

def plot_component(part):
    vertices, triangles = part.tessellate(0.05)
    x = [v.X for v in vertices]; y = [v.Y for v in vertices]; z = [v.Z for v in vertices]
    i = [t[0] for t in triangles]; j = [t[1] for t in triangles]; k = [t[2] for t in triangles]
    
    fig = go.Figure(data=[go.Mesh3d(x=x, y=y, z=z, i=i, j=j, k=k, color='lightgray', opacity=1.0, flatshading=True)])
    fig.update_layout(scene=dict(aspectmode='data'), title=f"3D Preview: {PART_NAME}")
    return fig

if __name__ == "__main__":
    part = build_component()
    analyze_and_export(part)
    # fig = plot_component(part)
    # fig.show()


--- PN_000669_v9 Analysis ---
Volume: 438.330 mm^3
Bounding Box: X[-7.75, 7.75], Y[-3.17, 0.00], Z[-5.00, 5.00]
Reference Volume: 438.966 mm^3
Volumetric Difference: 0.636 mm^3 (0.14%)
